# Notebook 05 (opcional) · El mismo problema con PyTorch: ciclo de entrenamiento explícito

| Redes Neuronales III | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-III` |


**Problema:** MNIST, 10 clases, red densa con Dropout y L2 (equivalente a `weight_decay`).

> **Alcance:** replicar la lógica de los notebooks anteriores en PyTorch, donde el ciclo de
> entrenamiento es explícito: `zero_grad()` → *forward* → *loss* → `backward()` → `step()`. El objetivo
> es comprobar que **los fundamentos no cambian de framework**.

---

## 1. Equivalencias Keras ↔ PyTorch

| Concepto | TensorFlow + Keras | PyTorch |
|---|---|---|
| Dropout 0.30 | `layers.Dropout(0.30)` | `nn.Dropout(0.30)` |
| Regularización L2 | `kernel_regularizer=regularizers.l2(1e-4)` | `weight_decay=1e-4` en el optimizador |
| Optimizador Adam | `keras.optimizers.Adam(...)` | `optim.Adam(...)` |
| Entrenamiento | `model.fit(...)` | Bucle `for` explícito |
| Early Stopping | `callbacks.EarlyStopping(...)` | Control manual sobre la loss de validación |
| Pérdida | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` (softmax implícito) |

**Nota:** en PyTorch la capa de salida entrega *logits*; `nn.CrossEntropyLoss` aplica `log_softmax`
internamente, por lo que **no** debe añadirse un `nn.Softmax` antes de la pérdida.

## 2. Entorno y datos

MNIST se descarga una sola vez como `mnist.npz` (~11 MB) y se lee con NumPy: así el notebook no
necesita importar TensorFlow y su huella de memoria queda reducida a PyTorch. El conjunto de
entrenamiento se conserva como tensor **uint8** y se normaliza lote a lote dentro del ciclo; Dev y
Test se evalúan por bloques directamente desde NumPy.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import pandas as pd
import urllib.request
import os
import gc

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Dispositivo:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.14.1+cpu
Dispositivo: cpu


In [2]:
SEED = 42

RUTA_NPZ = "mnist.npz"
if not os.path.exists(RUTA_NPZ):
    print("Descargando mnist.npz (~11 MB)...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
        RUTA_NPZ,
    )

with np.load(RUTA_NPZ) as datos:
    x_train_full, y_train_full = datos["x_train"], datos["y_train"]
    x_test_raw, y_test_np = datos["x_test"], datos["y_test"]

print("Train completo:", x_train_full.shape, "| Test:", x_test_raw.shape)

x_train_np, x_dev_np, y_train_np, y_dev_np = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)

# Train como tensor uint8 (47 MB); la normalización a [0,1] ocurre por lote.
train_dataset = TensorDataset(
    torch.tensor(x_train_np, dtype=torch.uint8),
    torch.tensor(y_train_np, dtype=torch.long),
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

del x_train_full, y_train_full
_ = gc.collect()

print("Train:", len(train_dataset), "| Dev:", len(x_dev_np), "| Test:", len(x_test_raw))

Descargando mnist.npz (~11 MB)...


Train completo: (60000, 28, 28) | Test: (10000, 28, 28)
Train: 48000 | Dev: 12000 | Test: 10000


## 3. Modelo con Dropout y L2 (weight_decay)

In [3]:
class RedMNISTRegularizada(nn.Module):
    """784 -> 256 -> 10 con Dropout(0.30); L2 entra por weight_decay del optimizador."""

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.30)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)


modelo_torch = RedMNISTRegularizada()
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_torch.parameters(), lr=0.001, weight_decay=1e-4)

print(modelo_torch)
total = sum(p.numel() for p in modelo_torch.parameters())
print()
print("Parámetros entrenables:", f"{total:,}")

RedMNISTRegularizada(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)

Parámetros entrenables: 203,530


## 4. Ciclo de entrenamiento explícito (3 épocas) con validación por época

Los cinco pasos aparecen visibles en el código. Al cierre de cada época se evalúa sobre Dev con
`modelo.eval()` y `torch.no_grad()`, y se registran loss y accuracy para graficar. La función
`evaluar_conjunto` procesa Dev y Test por bloques de 1 000 imágenes directamente desde NumPy.

In [4]:
def evaluar_conjunto(x_np, y_np, tam_lote=1000):
    """Pérdida y exactitud sobre un conjunto NumPy uint8, por bloques."""
    modelo_torch.eval()
    suma_loss, aciertos, total = 0.0, 0, 0
    with torch.no_grad():
        for inicio in range(0, len(x_np), tam_lote):
            xb = torch.tensor(
                x_np[inicio:inicio + tam_lote].reshape(-1, 784) / 255.0,
                dtype=torch.float32,
            )
            yb = torch.tensor(y_np[inicio:inicio + tam_lote], dtype=torch.long)
            out = modelo_torch(xb)
            suma_loss += criterio(out, yb).item() * len(yb)
            aciertos += int((out.argmax(1) == yb).sum())
            total += len(yb)
    return suma_loss / total, aciertos / total, aciertos


EPOCHS = 3
historial = {"train_loss": [], "train_acc": [], "dev_loss": [], "dev_acc": []}

for epoch in range(EPOCHS):
    modelo_torch.train()
    suma_loss, aciertos, total_obs = 0.0, 0, 0

    for imagenes, etiquetas in train_loader:
        xb = imagenes.float().reshape(len(imagenes), -1) / 255.0  # normaliza el lote

        optimizador.zero_grad()              # 1. limpiar gradientes acumulados
        salidas = modelo_torch(xb)           # 2. forward propagation
        error = criterio(salidas, etiquetas)  # 3. loss
        error.backward()                     # 4. backpropagation
        optimizador.step()                   # 5. actualización de pesos

        suma_loss += error.item() * len(etiquetas)
        aciertos += int((salidas.argmax(1) == etiquetas).sum())
        total_obs += len(etiquetas)

    dev_loss, dev_acc, _ = evaluar_conjunto(x_dev_np, y_dev_np)
    historial["train_loss"].append(suma_loss / total_obs)
    historial["train_acc"].append(aciertos / total_obs)
    historial["dev_loss"].append(dev_loss)
    historial["dev_acc"].append(dev_acc)
    print(f"Época {epoch+1} | train_loss {suma_loss/total_obs:.4f} "
          f"train_acc {aciertos/total_obs:.4f} | dev_loss {dev_loss:.4f} "
          f"dev_acc {dev_acc:.4f}")
    gc.collect()

tabla_torch = pd.DataFrame({
    "epoca": range(1, EPOCHS + 1),
    "train_loss": np.round(historial["train_loss"], 4),
    "train_acc": np.round(historial["train_acc"], 4),
    "dev_loss": np.round(historial["dev_loss"], 4),
    "dev_acc": np.round(historial["dev_acc"], 4),
})
print()
display(tabla_torch)

Época 1 | train_loss 0.3659 train_acc 0.8971 | dev_loss 0.1935 dev_acc 0.9468


Época 2 | train_loss 0.1761 train_acc 0.9489 | dev_loss 0.1371 dev_acc 0.9605


Época 3 | train_loss 0.1300 train_acc 0.9619 | dev_loss 0.1083 dev_acc 0.9690



,epoca,train_loss,train_acc,dev_loss,dev_acc
0,1,0.3659,0.8971,0.1935,0.9468
1,2,0.1761,0.9489,0.1371,0.9605
2,3,0.1300,0.9619,0.1083,0.9690


In [5]:
fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot(range(1, EPOCHS + 1), historial["train_loss"], marker="o", lw=2, label="Train loss")
ax.plot(range(1, EPOCHS + 1), historial["dev_loss"], marker="s", lw=2, ls="--", label="Dev loss")
ax.set_title("Figura 8. PyTorch - loss de entrenamiento y validación")
ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figIII_07_pytorch_curvas.png"); plt.show()
plt.close(fig); _ = gc.collect()

## 5. Evaluación final sobre Test

In [6]:
test_loss, test_acc, correctas = evaluar_conjunto(x_test_raw, y_test_np)

print("=== RESULTADO FINAL · PyTorch ===")
print(f"Accuracy en Test: {test_acc:.4f} ({int(correctas)}/{len(y_test_np)})")
print(f"Loss en Test    : {test_loss:.4f}")

=== RESULTADO FINAL · PyTorch ===
Accuracy en Test: 0.9704 (9704/10000)
Loss en Test    : 0.1029


## 6. Comparación con el enfoque Keras

| Aspecto | Keras (Notebooks 01-04) | PyTorch (este notebook) |
|---|---|---|
| Arquitectura | 784 → 128 → 10 y 512/256 regularizada | 784 → 256 → 10 con Dropout |
| Regularización L2 | `regularizers.l2(1e-4)` | `weight_decay=1e-4` |
| Backpropagation | Automática en `fit()` o `GradientTape` | Explícita en `error.backward()` |
| Actualización | Automática | Explícita en `optimizador.step()` |
| Evaluación | `predict` / evaluación por lotes | `eval()` + `no_grad()` por bloques |

La pequeña diferencia de exactitud entre frameworks se explica por arquitectura, épocas y muestreo por
lotes, **no** por el framework: ambos implementan el mismo ciclo matemático de aprendizaje.

## 7. Conclusiones del notebook

1. PyTorch hace visible cada paso del ciclo que Keras encapsula; dominar ambos niveles permite elegir
   la herramienta según el problema sin cambiar la teoría.
2. `weight_decay` es la forma idiomática de L2 en PyTorch; `nn.CrossEntropyLoss` incluye el softmax,
   por lo que la red entrega logits.
3. `modelo.eval()` y `torch.no_grad()` son obligatorios en evaluación: sin ellos, Dropout seguiría
   apagando unidades y los gradientes consumirían memoria inútilmente.

## 8. Referencias

- Stevens, E., Antiga, L., & Viehmann, T. (2020). *Deep Learning with PyTorch*. Manning Publications.
- LeCun, Y., Cortes, C., & Burges, C. J. C. (2010). *MNIST handwritten digit database*. http://yann.lecun.com/exdb/mnist/
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/